# Task 2 Unified Training Notebook (Local + Colab)

This notebook is rebuilt to be environment-agnostic and low-overhead.

How it works:
- Cell 1: intro
- Cell 2: **single source of truth** for environment and all config
- Cell 3 onward: all logic reads that config and runs the same workflow

Output artifacts:
- `model.pth`
- `metadata.json`
- learning curve plot (if generated)

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path
from typing import Any


def _is_colab_runtime() -> bool:
    # Robust Colab detection that avoids false positives on local machines
    # where google.colab might be installed as a dependency.
    if os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_GPU"):
        return True
    try:
        import google.colab  # type: ignore  # noqa: F401
    except Exception:
        return False
    return Path("/content").exists() and os.name != "nt"


def _load_dotenv_if_exists(path: Path) -> None:
    if not path.exists() or not path.is_file():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


def _env_bool(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "on"}:
        return True
    if normalized in {"0", "false", "no", "off"}:
        return False
    return default


def _env_int(name: str, default: int) -> int:
    value = os.environ.get(name)
    if value is None:
        return default
    try:
        return int(value)
    except ValueError:
        return default


def _env_float(name: str, default: float) -> float:
    value = os.environ.get(name)
    if value is None:
        return default
    try:
        return float(value)
    except ValueError:
        return default


def _search_repo_from_root(root: Path, max_depth: int = 6) -> Path | None:
    if not root.exists() or not root.is_dir():
        return None

    skip_dirs = {
        ".git",
        ".venv",
        "venv",
        "node_modules",
        "__pycache__",
        ".mypy_cache",
        ".pytest_cache",
        ".idea",
        ".vscode",
        "AppData",
    }

    root_parts = len(root.resolve().parts)
    for current, dirs, _files in os.walk(root):
        current_path = Path(current)
        depth = len(current_path.parts) - root_parts

        dirs[:] = [d for d in dirs if d not in skip_dirs]
        if depth > max_depth:
            dirs[:] = []
            continue

        marker = current_path / "task2_quality" / "training_pipeline.py"
        if marker.exists() and (current_path / "requirements.txt").exists():
            return current_path

    return None


def _resolve_local_repo_dir(preferred: str) -> Path:
    cwd = Path.cwd().resolve()

    direct_candidates: list[Path] = []
    if preferred.strip():
        direct_candidates.append(Path(preferred).expanduser().resolve())
    direct_candidates.extend([cwd, cwd.parent, cwd.parent.parent])

    for candidate in direct_candidates:
        if (candidate / "task2_quality" / "training_pipeline.py").exists() and (candidate / "requirements.txt").exists():
            return candidate

    home = Path.home().resolve()
    search_roots = [
        home / "Documents" / "Projects",
        home / "Documents",
        home / "Projects",
        home,
    ]

    for root in search_roots:
        found = _search_repo_from_root(root, max_depth=6)
        if found is not None:
            return found

    return cwd


for candidate in [Path.cwd() / ".env", Path.cwd().parent / ".env"]:
    _load_dotenv_if_exists(candidate)

runtime_is_colab = _is_colab_runtime()
mode = os.environ.get("TASK2_RUN_ENV", "auto").strip().lower()
if mode not in {"auto", "local", "colab"}:
    raise ValueError(f"TASK2_RUN_ENV must be one of auto/local/colab, got: {mode}")
if mode == "auto":
    mode = "colab" if runtime_is_colab else "local"

# Auto-correct accidental mode mismatch for smoother user experience.
if mode == "colab" and not runtime_is_colab:
    print("Info: TASK2_RUN_ENV=colab but runtime is local. Falling back to local mode.")
    mode = "local"
elif mode == "local" and runtime_is_colab:
    print("Info: TASK2_RUN_ENV=local but runtime is Colab. Falling back to colab mode.")
    mode = "colab"

is_colab = mode == "colab"

repo_url = os.environ.get("TASK2_REPO_URL", "https://github.com/graciesearle/AAI.git")
if is_colab:
    REPO_DIR = Path("/content/AAI")
    DATASET_HINT = Path("/content/task2_data")
    EXPORT_DIR = Path("/content/task2_export")
else:
    REPO_DIR = _resolve_local_repo_dir(os.environ.get("TASK2_REPO_DIR", ""))

    # Load repo-local .env after repo discovery so relative paths and credentials
    # come from the project config even if kernel cwd started elsewhere.
    _load_dotenv_if_exists(REPO_DIR / ".env")

    default_dataset = REPO_DIR / "FruitAndVegetableDataset"
    dataset_override = os.environ.get("TASK2_DATASET_DIR", "").strip()
    export_override = os.environ.get("TASK2_EXPORT_DIR", "").strip()

    if dataset_override:
        dataset_override_path = Path(dataset_override).expanduser()
        DATASET_HINT = dataset_override_path if dataset_override_path.is_absolute() else (REPO_DIR / dataset_override_path)
        DATASET_HINT = DATASET_HINT.resolve()
    else:
        DATASET_HINT = default_dataset

    if export_override:
        export_override_path = Path(export_override).expanduser()
        EXPORT_DIR = export_override_path if export_override_path.is_absolute() else (REPO_DIR / export_override_path)
        EXPORT_DIR = EXPORT_DIR.resolve()
    else:
        EXPORT_DIR = REPO_DIR / "task2_export"

SETTINGS: dict[str, Any] = {
    "mode": mode,
    "is_colab": is_colab,
    "repo_url": repo_url,
    "repo_dir": REPO_DIR,
    "dataset_hint": DATASET_HINT,
    "export_dir": EXPORT_DIR,
    "install_deps": _env_bool("TASK2_INSTALL_DEPS", True if is_colab else False),
    "allow_kaggle_download": _env_bool("TASK2_ALLOW_KAGGLE_DOWNLOAD", True if is_colab else False),
    "kaggle_dataset": os.environ.get(
        "TASK2_KAGGLE_DATASET",
        "muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten",
    ),
    "epochs": _env_int("TASK2_EPOCHS", 12),
    "batch_size": _env_int("TASK2_BATCH_SIZE", 32),
    "learning_rate": _env_float("TASK2_LEARNING_RATE", 0.001),
    "use_pretrained": _env_bool("TASK2_USE_PRETRAINED", True),
}

print(
    json.dumps(
        {
            "mode": SETTINGS["mode"],
            "is_colab": SETTINGS["is_colab"],
            "repo_dir": str(SETTINGS["repo_dir"]),
            "dataset_hint": str(SETTINGS["dataset_hint"]),
            "export_dir": str(SETTINGS["export_dir"]),
            "install_deps": SETTINGS["install_deps"],
            "allow_kaggle_download": SETTINGS["allow_kaggle_download"],
            "kaggle_dataset": SETTINGS["kaggle_dataset"],
            "epochs": SETTINGS["epochs"],
            "batch_size": SETTINGS["batch_size"],
            "learning_rate": SETTINGS["learning_rate"],
            "use_pretrained": SETTINGS["use_pretrained"],
        },
        indent=2,
    )
)

Info: TASK2_RUN_ENV=colab but runtime is local. Falling back to local mode.
{
  "mode": "local",
  "is_colab": false,
  "repo_dir": "C:\\Users\\jacob\\Documents\\Projects\\UNI\\AAI",
  "dataset_hint": "C:\\Users\\jacob\\Documents\\Projects\\UNI\\AAI\\FruitAndVegetableDataset",
  "export_dir": "C:\\Users\\jacob\\Documents\\Projects\\UNI\\AAI\\task2_export",
  "install_deps": true,
  "allow_kaggle_download": true,
  "kaggle_dataset": "muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten",
  "epochs": 12,
  "batch_size": 32,
  "learning_rate": 0.001,
  "use_pretrained": true
}


## Environment Bootstrap

This cell prepares repository location and optional dependency installation.

In [23]:
repo_dir = Path(SETTINGS["repo_dir"])

if SETTINGS["is_colab"]:
    if "<your-org-or-user>" in SETTINGS["repo_url"] and not repo_dir.exists():
        raise RuntimeError(
            "TASK2_REPO_URL is still a placeholder. Set a real repository URL in .env or environment."
        )
    if not repo_dir.exists():
        subprocess.run(["git", "clone", str(SETTINGS["repo_url"]), str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)
else:
    if not (repo_dir / "task2_quality" / "training_pipeline.py").exists():
        raise RuntimeError(
            f"Could not find task2_quality/training_pipeline.py under {repo_dir}. "
            "Run notebook from the AAI repository root or set TASK2_REPO_DIR."
        )

os.chdir(repo_dir)
print("Working directory:", Path.cwd())

if SETTINGS["install_deps"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    print("Installed requirements from requirements.txt")
else:
    print("Skipping dependency installation (TASK2_INSTALL_DEPS is false).")

Working directory: C:\Users\jacob\Documents\Projects\UNI\AAI
Installed requirements from requirements.txt


## Dataset Resolution

This cell resolves an existing dataset root first; if missing, it can download from Kaggle (depending on your config).

In [24]:
from getpass import getpass

dataset_hint = Path(SETTINGS["dataset_hint"]).expanduser()
dataset_slug = str(SETTINGS["kaggle_dataset"])
allow_download = bool(SETTINGS["allow_kaggle_download"])
has_kaggle_creds = bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
effective_allow_download = allow_download or has_kaggle_creds or SETTINGS["is_colab"]


def _score_dataset_dir(path: Path) -> int:
    try:
        children = [x for x in path.iterdir() if x.is_dir()]
    except Exception:
        return -1
    if len(children) < 2:
        return -1
    names = " ".join([c.name.lower() for c in children])
    bonus = 2 if ("healthy" in names or "rotten" in names) else 0
    return len(children) + bonus


def _detect_dataset_root(search_root: Path) -> Path | None:
    if not search_root.exists():
        return None
    if search_root.is_file():
        search_root = search_root.parent

    if _score_dataset_dir(search_root) >= 0:
        return search_root

    candidates = [p for p in search_root.rglob("*") if p.is_dir()]
    if not candidates:
        return None

    best = max(candidates, key=_score_dataset_dir)
    return best if _score_dataset_dir(best) >= 0 else None


def _prepare_kaggle_auth() -> None:
    username = os.environ.get("KAGGLE_USERNAME")
    key = os.environ.get("KAGGLE_KEY")

    if not username:
        username = input("Kaggle username: ").strip()
    if not key:
        key = getpass("Kaggle API key: ")

    if not username or not key:
        raise RuntimeError("Kaggle credentials are required for automatic dataset download.")

    os.environ["KAGGLE_USERNAME"] = username
    os.environ["KAGGLE_KEY"] = key

    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json = kaggle_dir / "kaggle.json"
    kaggle_json.write_text(json.dumps({"username": username, "key": key}), encoding="utf-8")
    os.chmod(kaggle_json, 0o600)


detected_dataset = _detect_dataset_root(dataset_hint)
if detected_dataset is None:
    if not effective_allow_download:
        raise RuntimeError(
            f"Dataset not found at {dataset_hint}. Set TASK2_DATASET_DIR to the exact dataset root, "
            "or enable TASK2_ALLOW_KAGGLE_DOWNLOAD=1, or provide both KAGGLE_USERNAME and KAGGLE_KEY."
        )

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    _prepare_kaggle_auth()

    download_root = dataset_hint if SETTINGS["is_colab"] else (dataset_hint.parent / "task2_data")
    download_root.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", dataset_slug, "-p", str(download_root), "--unzip"],
        check=True,
    )
    print("Dataset downloaded to:", download_root)

    detected_dataset = _detect_dataset_root(download_root)
    if detected_dataset is None:
        raise RuntimeError(
            f"Dataset download completed but no valid class-folder dataset was detected under {download_root}. "
            "Inspect extracted directories and set TASK2_DATASET_DIR to the correct folder."
        )

DATASET_DIR = detected_dataset
SETTINGS["dataset_dir"] = DATASET_DIR
print("Resolved DATASET_DIR:", DATASET_DIR)

Dataset downloaded to: C:\Users\jacob\Documents\Projects\UNI\AAI\task2_data
Resolved DATASET_DIR: C:\Users\jacob\Documents\Projects\UNI\AAI\task2_data\Fruit And Vegetable Diseases Dataset


## Train Model

This runs the production training script (`task2_quality/training_pipeline.py`).

In [25]:
cmd = [
    sys.executable,
    "task2_quality/training_pipeline.py",
    "--dataset-dir", str(SETTINGS["dataset_dir"]),
    "--model-version", "auto",
    "--epochs", str(SETTINGS["epochs"]),
    "--batch-size", str(SETTINGS["batch_size"]),
    "--learning-rate", str(SETTINGS["learning_rate"]),
]
cmd.append("--pretrained" if SETTINGS["use_pretrained"] else "--no-pretrained")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Training finished.")

Running: c:\Users\jacob\Documents\Projects\UNI\.venv\Scripts\python.exe task2_quality/training_pipeline.py --dataset-dir C:\Users\jacob\Documents\Projects\UNI\AAI\task2_data\Fruit And Vegetable Diseases Dataset --model-version auto --epochs 12 --batch-size 32 --learning-rate 0.001 --pretrained
Training finished.


## Resolve Outputs and Export

This identifies the latest semantic model version and exports files for Task 3 lifecycle operations.

In [26]:
from datetime import datetime, timezone


def _parse_semver(name: str):
    parts = name.split(".")
    if len(parts) != 3:
        return None
    try:
        return tuple(int(x) for x in parts)
    except ValueError:
        return None


model_root = Path(SETTINGS["repo_dir"]) / "models" / "produce-quality"
versions = []
for child in model_root.iterdir() if model_root.exists() else []:
    if child.is_dir():
        parsed = _parse_semver(child.name)
        if parsed is not None:
            versions.append((parsed, child.name))

if not versions:
    raise RuntimeError("No semantic model versions found under models/produce-quality after training.")

trained_version = sorted(versions, key=lambda x: x[0])[-1][1]
artifact_path = model_root / trained_version / "artifacts" / "model.pth"
plot_path = Path(SETTINGS["repo_dir"]) / "docs" / "task2" / f"task2_learning_curves_produce-quality_{trained_version}.png"

if not artifact_path.exists():
    raise RuntimeError(f"Expected artifact missing: {artifact_path}")

export_dir = Path(SETTINGS["export_dir"])
export_dir.mkdir(parents=True, exist_ok=True)

artifact_export_path = export_dir / "model.pth"
metadata_export_path = export_dir / "metadata.json"
artifact_export_path.write_bytes(artifact_path.read_bytes())
metadata_export_path.write_text(
    json.dumps(
        {
            "exported_at": datetime.now(timezone.utc).isoformat(),
            "mode": SETTINGS["mode"],
            "dataset_dir": str(SETTINGS["dataset_dir"]),
            "trained_version": trained_version,
            "artifact_source": str(artifact_path),
            "plot_path": str(plot_path),
        },
        indent=2,
    ),
    encoding="utf-8",
)

SETTINGS["trained_version"] = trained_version
SETTINGS["artifact_path"] = artifact_path
SETTINGS["plot_path"] = plot_path
SETTINGS["artifact_export_path"] = artifact_export_path
SETTINGS["metadata_export_path"] = metadata_export_path

print("trained_version:", trained_version)
print("artifact_source:", artifact_path)
print("artifact_export:", artifact_export_path)
print("metadata_export:", metadata_export_path)
if plot_path.exists():
    print("plot_path:", plot_path)

trained_version: 1.0.1
artifact_source: C:\Users\jacob\Documents\Projects\UNI\AAI\models\produce-quality\1.0.1\artifacts\model.pth
artifact_export: C:\Users\jacob\Documents\Projects\UNI\AAI\task2_export\model.pth
metadata_export: C:\Users\jacob\Documents\Projects\UNI\AAI\task2_export\metadata.json
plot_path: C:\Users\jacob\Documents\Projects\UNI\AAI\docs\task2\task2_learning_curves_produce-quality_1.0.1.png


## Retrieve Artifacts

In Colab, this downloads exported files. On local machines, it prints the saved paths.

In [27]:
if SETTINGS["is_colab"]:
    from google.colab import files
    files.download(str(SETTINGS["artifact_export_path"]))
    files.download(str(SETTINGS["metadata_export_path"]))
    if Path(SETTINGS["plot_path"]).exists():
        files.download(str(SETTINGS["plot_path"]))
else:
    print("Local mode output paths:")
    print("-", SETTINGS["artifact_export_path"])
    print("-", SETTINGS["metadata_export_path"])
    if Path(SETTINGS["plot_path"]).exists():
        print("-", SETTINGS["plot_path"])

Local mode output paths:
- C:\Users\jacob\Documents\Projects\UNI\AAI\task2_export\model.pth
- C:\Users\jacob\Documents\Projects\UNI\AAI\task2_export\metadata.json
- C:\Users\jacob\Documents\Projects\UNI\AAI\docs\task2\task2_learning_curves_produce-quality_1.0.1.png
